# 🩺 Health Alert — Rule-Based System v2.0

Sistem pakar berbasis aturan klinis untuk tiga kondisi: **Bronkitis**, **Anxiety**, **Alergi Dingin**.

## ✅ Perbaikan di v2.0

| Perbaikan | Detail |
| :--- | :--- |
| **Fix #1** | Normalisasi stress eksplisit via `stress_scale` (`'0-10'` / `'0-100'` / `'auto'`) |
| **Fix #2** | `spo2_confirmed` jadi parameter formal, bobot dinamis |
| **Fix #3** | Null-safe check untuk semua parameter opsional |
| **Fix #4** | Alergi Dingin pakai **ambang suhu personal** (`ambang_suhu_alergi`) |
| **Fix #5** | Saat suhu tidak tersedia → level `TIDAK_DIEVALUASI` (bukan AMAN palsu) |
| **Fix #6** | Pesan intervensi per kondisi × level |
| **Fix #7** | Profil kesehatan terpusat (HR warning/bahaya personal) |
| **Fix #8** | `spo2_confirmed` diteruskan dari orchestrator |

## 📋 Alur Sistem

```
Huawei Health → Health Sync → Health Connect
                          ↓
                   aplikasi Flutter
                          ↓
BMKG (suhu) ──→ Rule Engine ←── Profil Kesehatan
                          ↓
                   Alert + Intervensi
                          ↓
                Deduplikasi → Notifikasi
```

⚠️ **Disclaimer:** Ini bukan alat diagnosis. Threshold harus divalidasi dokter.

## 1. Setup

In [1]:
!pip install -q pandas numpy
import pandas as pd
import numpy as np
from typing import Optional, Dict, List

pd.set_option('display.width', 140)
pd.set_option('display.max_colwidth', 80)
print('Setup siap.')

Setup siap.


## 2. Utility — Normalisasi Skala

**PENTING:** parameter `stress_scale` wajib dispesifikasikan saat memanggil:

| Sumber | Skala | Nilai `stress_scale` |
| :--- | :--- | :--- |
| Huawei Health | 1–100 | `'0-100'` |
| Kaggle dataset | 1–10 | `'0-10'` |
| Tidak yakin | — | `'auto'` (default) |

In [2]:
def normalisasi_skala(nilai: Optional[float], skala_max: float) -> Optional[float]:
    if nilai is None or pd.isna(nilai):
        return None
    return float(nilai) / float(skala_max)


def normalisasi_input(data: dict) -> dict:
    """Normalisasi ke skala standar. Terima stress_scale eksplisit."""
    out = {}
    out['heart_rate'] = data.get('heart_rate')
    out['spo2'] = data.get('spo2')

    # Sleep quality: 0-1 atau 0-100
    sq = data.get('sleep_quality')
    if sq is not None and sq > 1:
        sq = normalisasi_skala(sq, 100)
    out['sleep_quality'] = sq

    # Stress: EXPLICIT scale
    st = data.get('stress_level')
    scale = data.get('stress_scale', 'auto')
    if st is not None:
        st = float(st)
        if scale == '0-100':
            st = st / 10.0
        elif scale == '0-10':
            pass
        else:  # auto
            if st > 10:
                st = st / 10.0
        st = max(0.0, min(10.0, st))
    out['stress_level_10'] = st
    out['stress_level_100'] = st * 10 if st is not None else None

    out['exercise_minutes'] = data.get('exercise_minutes')
    out['cycle_day'] = data.get('cycle_day')
    return out


def fase_siklus(cycle_day: Optional[int]) -> str:
    if cycle_day is None or pd.isna(cycle_day):
        return 'tidak_diketahui'
    if 1 <= cycle_day <= 5: return 'menstruasi'
    if 6 <= cycle_day <= 13: return 'folikular'
    if cycle_day == 14: return 'ovulasi'
    if 15 <= cycle_day <= 28: return 'luteal'
    return 'tidak_diketahui'


def level_dari_skor(score: int) -> str:
    if score >= 70: return 'BAHAYA'
    if score >= 45: return 'PERHATIAN'
    if score >= 20: return 'WASPADA'
    return 'AMAN'

print('Utility siap.')

Utility siap.


## 3. Profil Kesehatan — Threshold Personal

Threshold personal inilah yang membedakan sistemmu dari aplikasi generik. **Wajib disesuaikan dengan kondisi dan anjuran doktermu.**

In [3]:
# === UBAH SESUAI KONDISIMU ===
PROFIL_DEFAULT = {
    'nama': 'User',
    'kondisi': ['Bronkitis Kronis', 'Anxiety', 'Alergi Dingin'],
    'ambang_suhu_alergi': 20.0,    # °C — sesuaikan dengan observasi/dokter
    'hr_warning': 100,             # bpm — ambang waspada personal
    'hr_bahaya': 120,              # bpm — ambang bahaya personal
}

print('Profil default:')
for k, v in PROFIL_DEFAULT.items():
    print(f'  {k}: {v}')

Profil default:
  nama: User
  kondisi: ['Bronkitis Kronis', 'Anxiety', 'Alergi Dingin']
  ambang_suhu_alergi: 20.0
  hr_bahaya: 120


## 4. Rule — Bronkitis

**Perbaikan v2:**
- Null-safe: semua cek pakai `is not None`
- SpO₂ didukung `spo2_confirmed` (bobot lebih tinggi jika terkonfirmasi)
- HR pakai threshold personal dari profil

In [4]:
def skor_bronkitis(p: dict, spo2_confirmed: bool = False,
                   hr_warning: int = 100, hr_bahaya: int = 120) -> dict:
    score = 0
    reasons = []
    hr, spo2 = p['heart_rate'], p['spo2']
    sq, stress = p['sleep_quality'], p['stress_level_10']
    exercise, cycle = p['exercise_minutes'], p['cycle_day']

    # --- SpO2 (bobot terbesar) ---
    if spo2 is not None:
        if spo2 < 90:
            score += 40 if spo2_confirmed else 25
            reasons.append(f'SpO2 {spo2:.0f}% — hipoksemia' + ('' if spo2_confirmed else ' (perlu konfirmasi)'))
        elif spo2 < 92:
            score += 30 if spo2_confirmed else 18
            reasons.append(f'SpO2 {spo2:.0f}% — hipoksemia ringan' + ('' if spo2_confirmed else ' (perlu konfirmasi)'))
        elif spo2 < 94:
            score += 15
            reasons.append(f'SpO2 {spo2:.0f}% — borderline rendah')

    # --- Heart rate (personal threshold) ---
    if hr is not None:
        if hr > hr_bahaya:
            score += 25; reasons.append(f'HR {hr:.0f} bpm — takikardia berat')
        elif hr > hr_warning:
            score += 15; reasons.append(f'HR {hr:.0f} bpm — takikardia')

    # --- Kualitas tidur ---
    if sq is not None:
        if sq < 0.3:
            score += 15; reasons.append(f'Tidur sangat buruk ({sq:.2f})')
        elif sq < 0.5:
            score += 10; reasons.append(f'Tidur buruk ({sq:.2f})')

    # --- Stres ---
    if stress is not None:
        if stress >= 8:
            score += 10; reasons.append(f'Stres tinggi ({stress:.1f}/10) — bisa memperberat sesak')
        elif stress >= 7:
            score += 5

    # --- Olahraga saat gejala ---
    if exercise is not None and exercise > 60 and score >= 30:
        score += 10; reasons.append(f'Olahraga berat {exercise:.0f} mnt saat gejala — hindari')

    # --- Siklus ---
    if fase_siklus(cycle) == 'luteal' and score >= 20:
        score += 5; reasons.append('Fase luteal — sesak bisa memberat (progesteron)')

    score = min(int(score), 100)
    return {'kondisi': 'Bronkitis', 'skor': score,
            'level': level_dari_skor(score), 'alasan': reasons}

print('Rule bronkitis siap (v2, null-safe).')

Rule bronkitis siap (v2, null-safe).


## 5. Rule — Anxiety
HR pakai threshold personal. Olahraga protektif.

In [5]:
def skor_anxiety(p: dict, hr_warning: int = 96, hr_bahaya: int = 110) -> dict:
    score = 0
    reasons = []
    hr, sq = p['heart_rate'], p['sleep_quality']
    stress = p['stress_level_10']
    exercise, cycle = p['exercise_minutes'], p['cycle_day']

    if stress is not None:
        if stress >= 9:
            score += 40; reasons.append(f'Stres ekstrem ({stress:.1f}/10)')
        elif stress >= 8:
            score += 30; reasons.append(f'Stres sangat tinggi ({stress:.1f}/10)')
        elif stress >= 7:
            score += 20; reasons.append(f'Stres tinggi ({stress:.1f}/10)')
        elif stress >= 5:
            score += 10; reasons.append(f'Stres sedang ({stress:.1f}/10)')

    if hr is not None:
        if hr > hr_bahaya:
            score += 20; reasons.append(f'HR {hr:.0f} bpm — takikardia')
        elif hr > hr_warning:
            score += 12; reasons.append(f'HR {hr:.0f} bpm — di atas normal istirahat')

    if sq is not None:
        if sq < 0.3:
            score += 20; reasons.append(f'Tidur sangat buruk ({sq:.2f})')
        elif sq < 0.5:
            score += 12; reasons.append(f'Tidur buruk ({sq:.2f})')

    fase = fase_siklus(cycle)
    if fase == 'luteal':
        score += 10; reasons.append('Fase luteal — progesteron pro-kecemasan')
    elif fase == 'menstruasi':
        score += 5; reasons.append('Fase menstruasi')

    if exercise is not None:
        if exercise >= 30 and score > 0:
            score -= 10; reasons.append(f'Olahraga {exercise:.0f} mnt — protektif')
        elif exercise < 15:
            score += 5; reasons.append('Aktivitas fisik minim')

    score = max(0, min(int(score), 100))
    return {'kondisi': 'Anxiety', 'skor': score,
            'level': level_dari_skor(score), 'alasan': reasons}

print('Rule anxiety siap (v2, null-safe).')

Rule anxiety siap (v2, null-safe).


## 6. Rule — Alergi Dingin

**Perbaikan v2:**
- **Cek suhu PERTAMA.** Jika `None` → return `TIDAK_DIEVALUASI` (bukan AMAN palsu)
- Pakai **ambang suhu personal** (`ambang_suhu_alergi`)
- 3-tier: 10°C di bawah ambang / 5°C di bawah / di bawah ambang

In [6]:
def skor_alergi_dingin(p: dict, suhu_lingkungan: Optional[float] = None,
                       ambang_suhu_personal: float = 20.0,
                       hr_warning: int = 100, hr_bahaya: int = 110) -> dict:
    # === CEK SUHU PERTAMA (fix #2) ===
    if suhu_lingkungan is None:
        return {
            'kondisi': 'Alergi Dingin',
            'skor': 0,
            'level': 'TIDAK_DIEVALUASI',
            'alasan': ['Data suhu lingkungan tidak tersedia'],
            'confidence': 'LOW',
            'evaluated': False,
        }

    score = 0
    reasons = []
    hr = p['heart_rate']
    stress = p['stress_level_10']
    exercise, cycle = p['exercise_minutes'], p['cycle_day']

    # --- Suhu (personal threshold) ---
    if suhu_lingkungan <= ambang_suhu_personal - 10:
        score += 30
        reasons.append(f'Suhu {suhu_lingkungan:.0f}°C — sangat dingin (10°C di bawah ambang personal {ambang_suhu_personal:.0f}°C)')
    elif suhu_lingkungan <= ambang_suhu_personal - 5:
        score += 20
        reasons.append(f'Suhu {suhu_lingkungan:.0f}°C — dingin (di bawah ambang personal)')
    elif suhu_lingkungan <= ambang_suhu_personal:
        score += 10
        reasons.append(f'Suhu {suhu_lingkungan:.0f}°C — mendekati ambang personal')

    # --- HR (simpatis hiperaktif) ---
    if hr is not None:
        if hr > hr_bahaya:
            score += 25; reasons.append(f'HR {hr:.0f} bpm — simpatis hiperaktif')
        elif hr > hr_warning:
            score += 15; reasons.append(f'HR {hr:.0f} bpm — di atas normal')

    # --- Stres ---
    if stress is not None:
        if stress >= 8:
            score += 20; reasons.append(f'Stres sangat tinggi ({stress:.1f}/10)')
        elif stress >= 7:
            score += 12; reasons.append(f'Stres tinggi ({stress:.1f}/10)')

    # --- Olahraga (heat-cold) ---
    if exercise is not None and exercise > 45:
        score += 10; reasons.append(f'Olahraga {exercise:.0f} mnt — risiko heat-cold')

    # --- Siklus ---
    if fase_siklus(cycle) == 'luteal':
        score += 10; reasons.append('Fase luteal — sensitivitas dingin bisa meningkat')

    score = min(int(score), 100)
    return {'kondisi': 'Alergi Dingin', 'skor': score,
            'level': level_dari_skor(score), 'alasan': reasons,
            'confidence': 'HIGH', 'evaluated': True}

print('Rule alergi dingin siap (v2, suhu-first).')

Rule alergi dingin siap (v2, suhu-first).


## 7. Pesan Intervensi

Inti UX sistemmu — bukan cuma *alert*, tapi *apa yang harus dilakukan*.

| Level | Arti |
| :--- | :--- |
| AMAN | Tidak perlu tindakan |
| WASPADA | Preventive, lakukan sekarang sebelum memberat |
| PERHATIAN | Butuh perhatian, intervensi aktif |
| BAHAYA | Tindakan segera, eskalasi ke profesional jika perlu |

In [7]:
INTERVENSI = {
    'Bronkitis': {
        'AMAN': '',
        'WASPADA': 'Pantau napas. Hindari aktivitas berat hari ini.',
        'PERHATIAN': 'Duduk tegak. Latihan pursed-lip breathing. Siapkan inhaler/obat.',
        'BAHAYA': 'Istirahat total. Jika sesak memberat atau SpO2 <92% berulang, segera hubungi dokter/IGD.',
    },
    'Anxiety': {
        'AMAN': '',
        'WASPADA': 'Tarik napas 4-7-8 selama 3 menit. Minum air hangat.',
        'PERHATIAN': 'Grounding 5-4-3-2-1. Jauhkan diri dari pemicu. Minum air.',
        'BAHAYA': 'Duduk, minum air hangat, latihan pernapasan dalam. Hubungi support jika perlu.',
    },
    'Alergi Dingin': {
        'AMAN': '',
        'WASPADA': 'Pakai jaket sebelum keluar. Suhu saat ini {suhu}°C.',
        'PERHATIAN': 'Pakai lapisan hangat. Hindari paparan angin/AC langsung.',
        'BAHAYA': 'Tetap di dalam ruangan. Pakai pakaian hangat berlapis. Hindari kontak dengan benda dingin.',
    },
}


def ambil_intervensi(kondisi: str, level: str, suhu: Optional[float] = None) -> str:
    if level in ('AMAN', 'TIDAK_DIEVALUASI'):
        return ''
    template = INTERVENSI.get(kondisi, {}).get(level, '')
    if suhu is not None:
        return template.format(suhu=f'{suhu:.0f}')
    return template

print('Pesan intervensi siap.')

Pesan intervensi siap.


## 8. Orkestrator — Cek Semua Kondisi

Menggabungkan: normalisasi → 3 rule → intervensi → flag deduplikasi → DataFrame.

In [8]:
def cek_semua_kondisi(user_data: dict, profil: dict = None,
                      suhu_lingkungan: Optional[float] = None,
                      spo2_confirmed: bool = False,
                      kondisi_aktif: Optional[list] = None) -> pd.DataFrame:
    """
    user_data: dict dengan heart_rate, spo2, sleep_quality, stress_level, ...
    profil: dict threshold personal
    kondisi_aktif: list kondisi yang ingin dievaluasi (default: dari profil)
    """
    if profil is None:
        profil = PROFIL_DEFAULT

    p = normalisasi_input(user_data)

    hasil = []
    kondisi_aktif = kondisi_aktif or profil.get('kondisi', [])

    if 'Bronkitis Kronis' in kondisi_aktif or 'Bronkitis' in kondisi_aktif:
        hasil.append(skor_bronkitis(
            p, spo2_confirmed=spo2_confirmed,
            hr_warning=profil.get('hr_warning', 100),
            hr_bahaya=profil.get('hr_bahaya', 120),
        ))

    if 'Anxiety' in kondisi_aktif:
        hasil.append(skor_anxiety(p,
            hr_warning=max(96, profil.get('hr_warning', 100) - 4),
            hr_bahaya=profil.get('hr_bahaya', 120) - 10,
        ))

    if 'Alergi Dingin' in kondisi_aktif:
        hasil.append(skor_alergi_dingin(
            p, suhu_lingkungan=suhu_lingkungan,
            ambang_suhu_personal=profil.get('ambang_suhu_alergi', 20.0),
            hr_warning=profil.get('hr_warning', 100),
            hr_bahaya=profil.get('hr_bahaya', 110),
        ))

    # Confidence untuk bronkitis & anxiety berdasarkan jumlah parameter
    n_param = sum(1 for v in p.values() if v is not None and not pd.isna(v))
    conf = 'HIGH' if n_param >= 5 else ('MEDIUM' if n_param >= 3 else 'LOW')
    for h in hasil:
        h.setdefault('confidence', conf)
        h.setdefault('evaluated', True)
        h['intervensi'] = ambil_intervensi(
            h['kondisi'], h['level'], suhu_lingkungan
        )
        # Flag: level yang butuh alert
        h['butuh_alert'] = h['level'] in ('WASPADA', 'PERHATIAN', 'BAHAYA')

    df = pd.DataFrame([
        {
            'Kondisi': h['kondisi'],
            'Skor': h['skor'],
            'Level': h['level'],
            'Confidence': h['confidence'],
            'Butuh Alert': '✅' if h['butuh_alert'] else '—',
            'Alasan': ' | '.join(h['alasan']) or '-',
            'Intervensi': h['intervensi'] or '-',
        }
        for h in hasil
    ])
    return df

print('Orkestrator siap.')

Orkestrator siap.


## 9. Demo Skenario

**Catatan:** stress di demo ini pakai skala **0–10** (bukan 0–100), jadi kita berikan `stress_scale: '0-10'`.

In [9]:
skenario = {
    '1. Sehat total': {
        'heart_rate': 72, 'spo2': 98, 'sleep_quality': 0.85,
        'stress_level': 3, 'stress_scale': '0-10',
        'exercise_minutes': 45, 'cycle_day': 10
    },
    '2. Bronkitis flare (SpO2 rendah, unconfirmed)': {
        'heart_rate': 108, 'spo2': 89, 'sleep_quality': 0.30,
        'stress_level': 8, 'stress_scale': '0-10',
        'exercise_minutes': 0, 'cycle_day': 24
    },
    '3. Anxiety episode': {
        'heart_rate': 105, 'spo2': 97, 'sleep_quality': 0.35,
        'stress_level': 9, 'stress_scale': '0-10',
        'exercise_minutes': 10, 'cycle_day': 26
    },
    '4. Alergi dingin (suhu 5°C)': {
        'heart_rate': 112, 'spo2': 96, 'sleep_quality': 0.6,
        'stress_level': 7, 'stress_scale': '0-10',
        'exercise_minutes': 30, 'cycle_day': 22
    },
    '5. Kompleks — luteal + stres sedang (TANPA suhu)': {
        'heart_rate': 95, 'spo2': 95, 'sleep_quality': 0.45,
        'stress_level': 7, 'stress_scale': '0-10',
        'exercise_minutes': 20, 'cycle_day': 25
    },
}

for nama, data in skenario.items():
    suhu = 5.0 if 'dingin' in nama.lower() else None
    print(f'\n{"="*80}\n{nama}\n{"="*80}')
    display(cek_semua_kondisi(data, suhu_lingkungan=suhu, spo2_confirmed=False))


1. Sehat total


,Kondisi,Skor,Level,Confidence,Butuh Alert,Alasan,Intervensi
0,Bronkitis,0,AMAN,HIGH,—,-,-
1,Anxiety,0,AMAN,HIGH,—,-,-
2,Alergi Dingin,0,TIDAK_DIEVALUASI,LOW,—,Data suhu lingkungan tidak tersedia,-



2. Bronkitis flare (SpO2 rendah, unconfirmed)


,Kondisi,Skor,Level,Confidence,Butuh Alert,Alasan,Intervensi
0,Bronkitis,65,PERHATIAN,HIGH,✅,SpO2 89% — hipoksemia (perlu konfirmasi) | HR 108 bpm — takikardia | Tidur b...,Duduk tegak. Latihan pursed-lip breathing. Siapkan inhaler/obat.
1,Anxiety,69,PERHATIAN,HIGH,✅,Stres sangat tinggi (8.0/10) | HR 108 bpm — di atas normal istirahat | Tidur...,Grounding 5-4-3-2-1. Jauhkan diri dari pemicu. Minum air.
2,Alergi Dingin,0,TIDAK_DIEVALUASI,LOW,—,Data suhu lingkungan tidak tersedia,-



3. Anxiety episode


,Kondisi,Skor,Level,Confidence,Butuh Alert,Alasan,Intervensi
0,Bronkitis,40,WASPADA,HIGH,✅,HR 105 bpm — takikardia | Tidur buruk (0.35) | Stres tinggi (9.0/10) — bisa ...,Pantau napas. Hindari aktivitas berat hari ini.
1,Anxiety,79,BAHAYA,HIGH,✅,Stres ekstrem (9.0/10) | HR 105 bpm — di atas normal istirahat | Tidur buruk...,"Duduk, minum air hangat, latihan pernapasan dalam. Hubungi support jika perlu."
2,Alergi Dingin,0,TIDAK_DIEVALUASI,LOW,—,Data suhu lingkungan tidak tersedia,-



4. Alergi dingin (suhu 5°C)


,Kondisi,Skor,Level,Confidence,Butuh Alert,Alasan,Intervensi
0,Bronkitis,25,WASPADA,HIGH,✅,HR 112 bpm — takikardia | Fase luteal — sesak bisa memberat (progesteron),Pantau napas. Hindari aktivitas berat hari ini.
1,Anxiety,40,WASPADA,HIGH,✅,Stres tinggi (7.0/10) | HR 112 bpm — takikardia | Fase luteal — progesteron ...,Tarik napas 4-7-8 selama 3 menit. Minum air hangat.
2,Alergi Dingin,67,PERHATIAN,HIGH,✅,Suhu 5°C — sangat dingin (10°C di bawah ambang personal 20°C) | HR 112 bpm —...,Pakai lapisan hangat. Hindari paparan angin/AC langsung.



5. Kompleks — luteal + stres sedang (TANPA suhu)


,Kondisi,Skor,Level,Confidence,Butuh Alert,Alasan,Intervensi
0,Bronkitis,15,AMAN,HIGH,—,Tidur buruk (0.45),-
1,Anxiety,42,WASPADA,HIGH,✅,Stres tinggi (7.0/10) | Tidur buruk (0.45) | Fase luteal — progesteron pro-k...,Tarik napas 4-7-8 selama 3 menit. Minum air hangat.
2,Alergi Dingin,0,TIDAK_DIEVALUASI,LOW,—,Data suhu lingkungan tidak tersedia,-


## 10. Batch Test pada Dataset Publik (Kaggle)

Dataset publik memakai stress **1–10**. Kita kirim `stress_scale='0-10'`.

⚠️ Dataset ini **populasi umum**, bukan penderita 3 kondisi kronis. Distribusi AMAN >90% itu **wajar**. Fokusnya: pastikan sistem **tidak crash** dan **tidak over-alert**.

In [10]:
import kagglehub, os

def load_csv(path):
    for root, _, files in os.walk(path):
        for f in files:
            if f.endswith('.csv'):
                return pd.read_csv(os.path.join(root, f))

try:
    path = kagglehub.dataset_download('sabyasachisamadder/smart-watch-health-data-clean')
    df = load_csv(path).rename(columns={
        'Heart Rate (BPM)': 'heart_rate',
        'Blood Oxygen Level (%)': 'spo2',
        'Sleep Duration (hours)': 'sleep_hours',
        'Stress Level': 'stress_level',
        'Step Count': 'step_count'
    })
    for col in ['heart_rate', 'spo2', 'sleep_hours', 'stress_level', 'step_count']:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    df = df.dropna(subset=['heart_rate', 'spo2', 'sleep_hours', 'stress_level'])

    print(f'Rentang stress_level (Kaggle): {df["stress_level"].min():.1f} – {df["stress_level"].max():.1f}')

    df['sleep_quality'] = (df['sleep_hours'] / 8).clip(0, 1)
    df['exercise_minutes'] = (df['step_count'] / 100).clip(0, 90)

    hasil = {'Bronkitis': [], 'Anxiety': [], 'Alergi Dingin': []}
    for _, row in df.iterrows():
        d = {
            'heart_rate': row['heart_rate'], 'spo2': row['spo2'],
            'sleep_quality': row['sleep_quality'],
            'stress_level': row['stress_level'], 'stress_scale': '0-10',  # KUNCI
            'exercise_minutes': row['exercise_minutes'], 'cycle_day': None,
        }
        hasil['Bronkitis'].append(skor_bronkitis(normalisasi_input(d))['level'])
        hasil['Anxiety'].append(skor_anxiety(normalisasi_input(d))['level'])
        # Alergi dingin: tidak ada suhu → TIDAK_DIEVALUASI (skip)

    for kondisi, levels in hasil.items():
        if not levels:
            continue
        print(f'\nDistribusi {kondisi} pada {len(levels)} hari:')
        print(pd.Series(levels).value_counts())

except Exception as e:
    print(f'Batch test dilewati: {e}')

100%|██████████| 181k/181k [00:00<00:00, 23.3MB/s]

Extracting files...
Rentang stress_level (Kaggle): 1.0 – 10.0



Distribusi Bronkitis pada 9076 hari:
AMAN         8650
WASPADA       409
PERHATIAN      17
Name: count, dtype: int64

Distribusi Anxiety pada 9076 hari:
AMAN         5914
WASPADA      2731
PERHATIAN     431
Name: count, dtype: int64


## 11. Ekspor Konfigurasi untuk Aplikasi Flutter

File `rule_config.json` ini yang akan kamu baca di aplikasi Dart. **Jangan hardcode threshold** — baca dari JSON.

In [11]:
import json

konfigurasi = {
    'versi': '2.0.0',
    'level_threshold': {'AMAN': 0, 'WASPADA': 20, 'PERHATIAN': 45, 'BAHAYA': 70},
    'profil_default': PROFIL_DEFAULT,
    'bronkitis': {
        'spo2': {'<90_unconfirmed': 25, '<90_confirmed': 40,
                 '<92_unconfirmed': 18, '<92_confirmed': 30, '<94': 15},
        'heart_rate': {'>hr_bahaya': 25, '>hr_warning': 15},
        'sleep_quality': {'<0.3': 15, '<0.5': 10},
        'stress': {'>=8': 10, '>=7': 5},
        'exercise': {'>60_dan_skor>=30': 10},
        'luteal': {'skor>=20': 5},
    },
    'anxiety': {
        'stress': {'>=9': 40, '>=8': 30, '>=7': 20, '>=5': 10},
        'heart_rate': {'>hr_bahaya': 20, '>hr_warning': 12},
        'sleep_quality': {'<0.3': 20, '<0.5': 12},
        'luteal': 10, 'menstruasi': 5,
        'exercise': {'>=30': -10, '<15': 5},
    },
    'alergi_dingin': {
        'suhu_personal': {'<=ambang-10': 30, '<=ambang-5': 20, '<=ambang': 10},
        'heart_rate': {'>hr_bahaya': 25, '>hr_warning': 15},
        'stress': {'>=8': 20, '>=7': 12},
        'exercise': {'>45': 10},
        'luteal': 10,
    },
    'intervensi': INTERVENSI,
    'deduplikasi': {
        'kirim_ulang_setelah_menit': 60,
        'kirim_ulang_jika_level_berubah': True,
        'jangan_kirim_jika_level_AMAN': True,
    },
    'sumber_data': {
        'huawei_stress_scale': '0-100',
        'kaggle_stress_scale': '0-10',
        'suhu_sumber': 'BMKG API',
    },
    'catatan': 'Threshold klinis berdasarkan literatur. Wajib divalidasi dokter sebelum dipakai untuk keputusan medis.'
}

with open('rule_config.json', 'w', encoding='utf-8') as f:
    json.dump(konfigurasi, f, indent=2, ensure_ascii=False)

print('Konfigurasi tersimpan sebagai rule_config.json')
print(f'Versi: {konfigurasi["versi"]}')
print(f'Total key top-level: {len(konfigurasi)}')

Konfigurasi tersimpan sebagai rule_config.json
Versi: 2.0.0
Total key top-level: 10


## 12. Langkah Selanjutnya (Bukan di Notebook Ini)

### ➡️ Yang HARUS kamu lakukan minggu ini (Mode Observasi)
1. **Matikan notifikasi.** Jangan aktifkan alert dulu.
2. Jalankan `cek_semua_kondisi()` setiap 15 menit via `Timer.periodic` di Flutter.
3. Simpan setiap output ke SQLite.
4. Setiap hari, tulis **ground truth** manual: "hari ini aku cemas? sesak? kedinginan?"

### ➡️ Yang HARUS kamu lakukan minggu ke-3 (Kalibrasi)
Setelah 2 minggu data, hitung di Python:
```python
# Rata-rata HR saat kamu merasa AMAN → personal HR baseline
baseline_hr = df_aman['heart_rate'].mean()
profil['hr_warning'] = int(baseline_hr + 15)
profil['hr_bahaya'] = int(baseline_hr + 30)
```

### ➡️ Minggu 4-6 (Integrasi Flutter + BMKG)
- Porting fungsi `skor_bronkitis`, `skor_anxiety`, `skor_alergi_dingin` ke Dart
- Baca `rule_config.json` dari assets
- Panggil BMKG API → isi `suhu_lingkungan`
- `flutter_local_notifications` + dedup state (60 menit)

### ❌ Yang JANGAN dilakukan sekarang
- **Jangan training ML** — kamu belum punya data berlabel personal
- **Jangan aktifkan notifikasi** sebelum kalibrasi
- **Jangan percaya threshold default** — semuanya perlu disesuaikan